# TOKATÇI — Q(s,a) / action-value (Kaggle)

Amaç: **state + seçilen aksiyon → kazanma olasılığı**.

Önemli sınırlama:
- Replay’de yalnızca seçilmiş aksiyonun outcome’u var.
- Bu yüzden bu model **selected-action Q** olarak eğitilir.
- Search’te BC top-K adaylarını yeniden sıralamak için kullanılacak.
- Full Q / afterstate Q, engine’de her aday aksiyon simüle edilerek sonra üretilecek.

Girdi:
- state: 50
- option: 48
- actor/target card IDs

Çıktı:
- seçilen aksiyonun win logiti


In [ ]:
from pathlib import Path
import numpy as np, torch
from torch import nn, optim

INPUT=Path('/kaggle/input')
DATA=next(INPUT.rglob('dataset_tokatci_b1.npz'),None)
if DATA is None: raise FileNotFoundError('dataset_tokatci_b1.npz bulunamadı')
OUT=Path('/kaggle/working/model_tokatci_q.npz')
print(DATA, torch.__version__, 'cuda', torch.cuda.is_available())


In [ ]:
d=np.load(DATA)
state=d['state'].astype(np.float32)             # N,50
options=d['options'].astype(np.float32)         # N,40,48
card_ids=d['card_ids'].astype(np.int64)         # N,40,2
mask=d['mask'].astype(np.float32)               # N,40
label=d['label'].astype(np.int64)               # N
y=d['game_win'].astype(np.float32)              # N
w=d['sample_weight'].astype(np.float32)
split=d['split'].astype(np.int8)
N=len(state)
rows=np.arange(N)
opt_sel=options[rows,label]                     # N,48
cid_sel=card_ids[rows,label]                    # N,2
train_idx=np.nonzero(split==0)[0]; val_idx=np.nonzero(split==1)[0]
print('N',N,'train',len(train_idx),'val',len(val_idx),'win',y.mean(),'option',opt_sel.shape)


In [ ]:
# Normalizers: state train; selected option train.
s_mean=state[train_idx].mean(0); s_std=state[train_idx].std(0)
weak=s_std<1e-4; s_mean[weak]=0; s_std[weak]=1
o_mean=opt_sel[train_idx].mean(0); o_std=opt_sel[train_idx].std(0)
weak=o_std<1e-4; o_mean[weak]=0; o_std[weak]=1
print('norms',s_mean.shape,o_mean.shape)


In [ ]:
class QNet(nn.Module):
 def __init__(self, state_dim=50, option_dim=48, embed=32, hidden=128, dropout=.1):
  super().__init__()
  self.emb=nn.Embedding(2048,embed,padding_idx=0)
  self.state_proj=nn.Linear(state_dim,64)
  self.option_proj=nn.Linear(option_dim+2*embed,64)
  self.net=nn.Sequential(nn.Linear(128,hidden),nn.ReLU(),nn.Dropout(dropout),nn.Linear(hidden,64),nn.ReLU(),nn.Dropout(dropout),nn.Linear(64,1))
 def forward(self,s,o,cid):
  st=torch.relu(self.state_proj(s))
  actor=self.emb(cid[:,0]); target=self.emb(cid[:,1])
  op=torch.relu(self.option_proj(torch.cat([o,actor,target],dim=-1)))
  return self.net(torch.cat([st,op],dim=-1)).squeeze(-1)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
net=QNet().to(device); opt=optim.AdamW(net.parameters(),lr=1e-3,weight_decay=1e-4)
loss_fn=nn.BCEWithLogitsLoss(reduction='none')
print('device',device,'params',sum(p.numel() for p in net.parameters()))


In [ ]:
def metrics(yy,logit):
 p=1/(1+np.exp(-logit)); bce=float(np.mean(np.logaddexp(0,logit)-yy*logit)); brier=float(np.mean((p-yy)**2)); acc=float(((p>=.5)==yy).mean())
 order=np.argsort(p); ranks=np.empty_like(order,dtype=float); ranks[order]=np.arange(1,len(p)+1); n1=yy.sum(); n0=len(yy)-n1
 auc=float((ranks[yy==1].sum()-n1*(n1+1)/2)/(n1*n0)) if n1>0 and n0>0 else .5
 return bce,acc,auc,brier

def infer(idx,chunk=4096):
 net.eval(); out=[]
 with torch.no_grad():
  for st in range(0,len(idx),chunk):
   ii=idx[st:st+chunk]
   ss=torch.from_numpy((state[ii]-s_mean)/s_std).to(device)
   oo=torch.from_numpy((opt_sel[ii]-o_mean)/o_std).to(device)
   cc=torch.from_numpy(cid_sel[ii]).to(device)
   out.append(net(ss,oo,cc).cpu().numpy())
 return np.concatenate(out)


In [ ]:
best=(1e9,None,0); bad=0; patience=10; bs=1024
for epoch in range(1,41):
 net.train(); order=train_idx.copy(); np.random.shuffle(order)
 for st in range(0,len(order),bs):
  ii=order[st:st+bs]
  ss=torch.from_numpy((state[ii]-s_mean)/s_std).to(device,non_blocking=True)
  oo=torch.from_numpy((opt_sel[ii]-o_mean)/o_std).to(device,non_blocking=True)
  cc=torch.from_numpy(cid_sel[ii]).to(device,non_blocking=True)
  yy=torch.from_numpy(y[ii]).to(device,non_blocking=True)
  ww=torch.from_numpy(w[ii]).to(device,non_blocking=True)
  opt.zero_grad(set_to_none=True)
  loss=(loss_fn(net(ss,oo,cc),yy)*ww).sum()/ww.sum().clamp_min(1e-8)
  loss.backward(); opt.step()
 logit=infer(val_idx); m=metrics(y[val_idx],logit)
 print(f'epoch {epoch:2d} val bce={m[0]:.4f} acc={m[1]:.4f} auc={m[2]:.4f} brier={m[3]:.4f}')
 if m[0] < best[0]:
  best=(m[0],{k:v.detach().cpu().clone() for k,v in net.state_dict().items()},epoch); bad=0
 else:
  bad+=1
  if bad>=patience: print('early stop'); break
net.load_state_dict(best[1]); print('best',best[2],metrics(y[val_idx],infer(val_idx)))


In [ ]:
arrays={k:v.detach().cpu().numpy() for k,v in net.state_dict().items()}
arrays['state_mean']=s_mean.astype(np.float32); arrays['state_std']=s_std.astype(np.float32)
arrays['option_mean']=o_mean.astype(np.float32); arrays['option_std']=o_std.astype(np.float32)
np.savez_compressed(OUT,**arrays)
print('wrote',OUT,OUT.stat().st_size/1024,'KB')


## Search kullanımı

Q modeli **tüm legal option’ları** skorlayabilir:

1. BC ile top-K aday.
2. Her aday için Q(s,a).
3. `Q + λ * BC_logit` ile yeniden sırala.
4. Q farkı düşükse search/value; lethal varsa direkt.

Bu model selected-action Q olduğu için unselected aksiyonlarda gürültülü olabilir; top-K dışında kullanma.
